In [0]:
%python
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.table(f"{catalog}.bronze.orders")
#s1
df = df.withColumn('customer_id', trim(upper(col('customer_id')))).\
    withColumn('order_id', trim(upper(col('order_id')))).\
    withColumn('store_id', trim(upper(col('store_id')))).\
    withColumn('items', transform(col('items'), lambda x: x.withField('product_id', trim(upper(x['product_id'])))))
#s2
df = df.withColumn('order_status', regexp_replace(trim(upper(col('order_status'))), 'CANCELED', 'CANCELLED')).\
    filter(col('order_status').isin('PLACED', 'DELIVERED', 'CANCELLED'))
#s3
df = df.withColumn('payment.method', trim(upper(col('payment.method'))))
#s5 
df = df.withColumn('order_ts', expr("try_to_timestamp(regexp_replace(order_ts, 'T', ' '), 'yyyy-MM-dd HH:mm:ss')")).\
    withColumn('order_date', to_date(col('order_ts')))
#s6
df = df.withColumn('updated_at', expr("try_to_timestamp(regexp_replace(updated_at, 'T', ' '), 'yyyy-MM-dd HH:mm:ss')"))
# s7-s10: line-level fields (quantity, unit_price, discount, line_number)
# are derived in c3 after posexplode, so they are not computed here.


In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy('order_id').orderBy(col('updated_at').desc())
df = df.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')


In [0]:
# c3 – Validation: route each order to silver.orders or silver.quarantine_orders
# Only the latest version of each order (output of c2) is validated.
# An order is rejected if it breaks ANY rule; the rejection reason is the
# lowest-numbered broken rule.  A rejected order goes to quarantine as a
# whole — none of its lines reach silver.orders.  An order is in
# silver.orders OR silver.quarantine_orders, never both.

# ── Reference data for lookup validation (R2, R3, R6) ───────────────
customers_ref = (spark.read.table(f"{catalog}.bronze.customers")
    .select(trim(upper(col("customer_id"))).alias("customer_id")).distinct())
stores_ref = (spark.read.table(f"{catalog}.bronze.stores")
    .select(trim(upper(col("store_id"))).alias("store_id")).distinct())
products_ref = (spark.read.table(f"{catalog}.bronze.products")
    .select(trim(upper(col("product_id"))).alias("product_id")).distinct())

# ── Order-level reference joins (left join → null = unknown) ────────
df = (df
    .join(broadcast(customers_ref.withColumnRenamed("customer_id", "_ref_customer")),
          col("customer_id") == col("_ref_customer"), "left")
    .join(broadcast(stores_ref.withColumnRenamed("store_id", "_ref_store")),
          col("store_id") == col("_ref_store"), "left"))

# ── Order-level rules R1-R5 ─────────────────────────────────────────
df = (df
    .withColumn("r1", when(col("customer_id").isNull()
                          | (trim(col("customer_id")) == lit("")), 1).otherwise(0))
    .withColumn("r2", when((col("r1") == 0) & col("_ref_customer").isNull(), 1).otherwise(0))
    .withColumn("r3", when(col("_ref_store").isNull(), 1).otherwise(0))
    .withColumn("r4", when(col("order_ts").isNull(), 1).otherwise(0))
    .withColumn("r5", when(col("items").isNull() | (size(col("items")) <= 0), 1).otherwise(0)))

# ── Line-level rules R6-R8 (explode items, check each line) ─────────
line_checks = (
    df
    .select("order_id", posexplode(col("items")))
    .join(broadcast(products_ref.withColumnRenamed("product_id", "_ref_prod")),
          trim(upper(col("col.product_id"))) == col("_ref_prod"), "left")
    .withColumn("r6", when(col("_ref_prod").isNull(), 1).otherwise(0))
    .withColumn("_qty_num", expr("try_cast(`col`.qty as double)"))
    .withColumn("r7", when(col("col.qty").isNull()
                          | col("_qty_num").isNull()
                          | (col("_qty_num") <= 0)
                          | (col("_qty_num") % 1 != 0), 1).otherwise(0))
    .withColumn("r8", when(col("col.unit_price").isNull()
                          | (col("col.unit_price") <= 0), 1).otherwise(0))
    .groupBy("order_id")
    .agg(max(col("r6")).alias("r6"),
         max(col("r7")).alias("r7"),
         max(col("r8")).alias("r8")))

# ── Merge line-level checks back ────────────────────────────────────
df = (df.join(line_checks, "order_id", "left")
        .fillna(0, subset=["r6", "r7", "r8"]))

# ── Determine rejection reason (lowest-numbered broken rule) ───────
df = df.withColumn("rejection_reason",
    when(col("r1") == 1, "R1-MISSING_CUSTOMER")
    .when(col("r2") == 1, "R2-UNKNOWN_CUSTOMER")
    .when(col("r3") == 1, "R3-UNKNOWN_STORE")
    .when(col("r4") == 1, "R4-INVALID_TIMESTAMP")
    .when(col("r5") == 1, "R5-EMPTY_ORDER")
    .when(col("r6") == 1, "R6-UNKNOWN_PRODUCT")
    .when(col("r7") == 1, "R7-INVALID_QUANTITY")
    .when(col("r8") == 1, "R8-INVALID_PRICE")
    .otherwise(lit(None)))

# ── Split: valid → silver.orders, rejected → silver.quarantine_orders
_helper_cols = ["r1", "r2", "r3", "r4", "r5",
                "r6", "r7", "r8", "_ref_customer", "_ref_store"]

valid_df  = df.filter(col("rejection_reason").isNull()).drop(*_helper_cols)
reject_df = df.filter(col("rejection_reason").isNotNull()).drop(*_helper_cols)

# Silver — order-level (one row per valid order)
silver_orders = (
    valid_df
    .select("order_id", "customer_id", "store_id",
            "order_status", "order_ts", "order_date",
            "updated_at",
            col("`payment.method`").alias("payment_method"),
            col("delivery.promised_mins").cast("int").alias("promised_mins"),
            col("delivery.actual_mins").cast("int").alias("actual_mins"),
            col("delivery.delivery_fee").cast("decimal(10,2)").alias("delivery_fee")))

# Silver — line-level (one row per line item)
silver_order_items = (
    valid_df
    .select("order_id", posexplode(col("items")))
    .withColumn("line_number",      col("pos") + 1)
    .withColumn("product_id",       col("col.product_id"))
    .withColumn("quantity",         col("col.qty").cast("int"))
    .withColumn("unit_price",       col("col.unit_price").cast("decimal(10,2)"))
    .withColumn("discount_amount", coalesce(col("col.discount"), lit(0)).cast("decimal(10,2)"))
    .withColumn("gross_amount",     (col("quantity").cast("decimal(10,2)") * col("unit_price")).cast("decimal(10,2)"))
    .withColumn("net_amount",       (col("gross_amount") - col("discount_amount")).cast("decimal(10,2)"))
    .drop("pos", "col"))

# Quarantine — rejected orders as a whole (items array intact)
quarantine_orders = reject_df

# ── Incremental maintenance with MERGE INTO (idempotent) ────────────
def merge_or_create(table_full, source_df, merge_on):
    """Create on first run; MERGE INTO on subsequent runs.
    WHEN NOT MATCHED BY SOURCE DELETE removes orders that moved between
    valid and rejected states, keeping the two tables mutually exclusive.
    If the table exists but has a different schema (one-time migration),
    it is recreated with the new schema."""
    if not spark.catalog.tableExists(table_full):
        source_df.write.format("delta").saveAsTable(table_full)
    else:
        existing_cols = set(spark.read.table(table_full).columns)
        source_cols  = set(source_df.columns)
        if existing_cols != source_cols:
            source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_full)
        else:
            source_df.createOrReplaceTempView("merge_src")
            spark.sql(f"""
                MERGE INTO {table_full} AS t
                USING merge_src AS s
                ON {merge_on}
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
                WHEN NOT MATCHED BY SOURCE THEN DELETE
            """)

merge_or_create(f"{catalog}.silver.orders", silver_orders,
                "t.order_id = s.order_id")
merge_or_create(f"{catalog}.silver.order_items", silver_order_items,
                "t.order_id = s.order_id AND t.line_number = s.line_number")
merge_or_create(f"{catalog}.silver.quarantine_orders", quarantine_orders,
                "t.order_id = s.order_id")

print(f"silver.orders:          {spark.read.table(f'{catalog}.silver.orders').count()} orders")
print(f"silver.order_items:      {spark.read.table(f'{catalog}.silver.order_items').count()} line items")
print(f"silver.quarantine_orders: {spark.read.table(f'{catalog}.silver.quarantine_orders').count()} orders")

In [0]:
%sql
select * from fk_1368.bronze.orders
limit 1